# Phase 4b: Step 1 — Two-Way Fixed Effects

**ECON 499R — AI Impact on Renewables**  
**Date:** 2026-11-02  
**Driver:** `code/R/04b_step1_twfe.R --date 2026-11-02`  
**Results:** `results/2026-11-02/h1/`, `results/2026-11-02/tables/`, `results/2026-11-02/figures/`

**NAICS 518210 caveat:** Georgia ranks 2nd in `ai_dc_per1000` in 2025 (5.03, behind WA 6.48), above VA (4.86). This likely reflects colocation and cloud hosting, not AI-specific compute. Treat Georgia's high rank as scope noise in the main variable, not as a clean AI signal.

---
## 0. Reproduce gate

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{manuf} + \text{price} + \text{pop growth} + \mu_i + \lambda_t$  
**Tests:** match `econ499r_starter/reference/prelim_step1_slx.csv` within rounding.

| Outcome | Ref a₁ | Got a₁ | Diff | Result |
|---|---|---|---|---|
| ln_ec | −0.001565 | −0.001565 | 0.000000 | **PASS** |
| ln_ee | +0.002206 | +0.002206 | 0.000000 | **PASS** |
| renew_share | −0.010129 | −0.010129 | 0.000000 | **PASS** |
| solar_share | −0.000059 | −0.000059 | 0.000000 | **PASS** |
| new_renew_GW | +0.075995 | +0.075995 | 0.000000 | **PASS** |

All five match exactly. Proceeding to main analysis.

In [ ]:
setwd('/Users/shunji/Desktop/EMORY/ECON 499R/AI impact on Renewables')
library(data.table); library(ggplot2)

h1_core  <- fread('results/2026-11-02/h1/h1_core.csv')
h1_extra <- fread('results/2026-11-02/h1/h1_extras.csv')
se_check <- fread('results/2026-11-02/h1/se_check.csv')
nonlin   <- fread('results/2026-11-02/h1/nonlinearity.csv')
tp       <- fread('results/2026-11-02/h1/turning_points.csv')
het      <- fread('results/2026-11-02/h1/heterogeneity.csv')
dyn      <- fread('results/2026-11-02/h1/dynamics.csv')
cat('All result files loaded.\n')

---
## 1. H1 Core — with and without price control

**Equation:** $Y_{it} = a_1 \cdot \text{AI}_{i,t-1} + \ln\text{GDP/pc} + \text{price} + \text{tech share} + \text{manuf} + \text{pop growth} + \mu_i + \lambda_t$  
**Tests:** Is AI data-center concentration associated with consumption, efficiency, or renewable share?  
**N = 550** (50 states × 11 years). SE clustered by state. Winsorized 1/99.

In [ ]:
cols_show <- c('outcome','controls','estimate','se','pval','N','within_r2')
print(h1_core[, ..cols_show])

**Reading:**
- **ln_ec (consumption):** a₁ = −0.0022 (ctrl5), p = 0.704. Does not support H1. Adding tech_share to controls modestly shifts the coefficient from −0.0016 to −0.0022 but remains insignificant — tech_share is positively correlated with both AI and efficiency, so its inclusion absorbs some positive confounding.
- **ln_ee (efficiency):** a₁ = +0.0031, p = 0.567. Insignificant. Directionally positive (states with more AI data centers are slightly more efficient), but imprecise.
- **renew_share:** a₁ = −0.0103, p = 0.103. Borderline. Negative sign is opposite to the blessing direction. Does not support H1 at the 10% level under clustered SEs.
- **Price control:** The ctrl4 vs ctrl5 comparison shows minimal sensitivity — coefficients shift by <0.001 for all three outcomes. Price is not acting as a strong bad control in this sample.

---
## 2. H1 Extras

**Same equation, additional outcomes.**  
**Falsification:** coal_share should be null if the AI effect is specific to renewables.

In [ ]:
print(h1_extra[, .(outcome, estimate, se, pval, N, within_r2)])

**Reading:**
- **new_renew_mw: a₁ = +74.5 MW, se = 31.5, p = 0.022.** Only significant result. A one-unit increase in AI/1000 workers is associated with ~75 MW more new renewable capacity added that year. This is the clearest positive signal in the H1 battery.
- **solar_share:** a₁ = +0.0002, p = 0.955. Null.
- **wind_share:** a₁ = −0.0085, p = 0.222. Null.
- **fossil_to_renew:** a₁ = +0.95, p = 0.110. Wrong sign for H1 (H1 requires a₁ < 0). Does not support H1.
- **coal_share:** a₁ = +0.0019, p = 0.808. Null — as expected for a falsification check.
- **co2_per_mwh_t (2015–2024):** a₁ = +0.0022, p = 0.731. Null. Wrong sign direction but not significant.

---
## 3. SE Check: Clustered vs Driscoll-Kraay

In [ ]:
print(se_check[, .(outcome, vcov_type, estimate, se, pval)])

**Reading:** Driscoll-Kraay SEs are substantially *smaller* than state-clustered SEs (~4× smaller for ln_ee and renew_share). This is unexpected — DK accounts for serial correlation *and* cross-sectional dependence, so it usually produces wider CIs. The smaller DK SEs here likely reflect that cross-sectional correlations are not large after removing year FE.

Under DK:
- **ln_ee: p = 0.028** (significant at 5%)
- **renew_share: p = 0.001** (significant at 1%)
- ln_ec: p = 0.172 (still insignificant)

**DK caveat — sensitivity check only (T = 11):** With only 11 time periods, the Bartlett kernel in Driscoll-Kraay defaults to a lag window of approximately 2, which is too short to fully capture serial correlation. DK results are reported as a sensitivity check only. The baseline for every H1 conclusion is clustered by state (CRV1). Do not cite DK significance as primary evidence for H1.

---
## 4. Nonlinearity (AI + AI²)

**Equation:** $Y_{it} = b_1 \cdot \text{AI}_{i,t-1} + b_2 \cdot \text{AI}^2_{i,t-1} + \text{controls} + \mu_i + \lambda_t$  
**Turning point:** $\text{AI}^* = -b_1 / (2 b_2)$ with delta-method SE.

In [ ]:
print(tp[, .(outcome, b1, b2, turning_point, tp_se, tp_lo, tp_hi, ai_min, ai_max,
             inside_range, pct_state_yrs_past_tp, lm_opposite_signs)])

**Reading — corrected Lind-Mehlum (delta-method, updated 2026-11-02):**

The original analysis used a sign-only check (opposite slopes at min/max). The correct Lind-Mehlum test requires the slope `b₁ + 2·b₂·AI` to be statistically significant (one-sided, |t| > 1.645) at **both** the minimum and maximum of lagged AI, with SE from delta-method including cov(b₁, b₂). Results from `results/2026-11-02/h1/lm_proper.csv`:

| Outcome | b₂ | p(b₂) | t(slope at min) | t(slope at max) | LM verdict |
|---|---|---|---|---|---|
| ln_ec | −0.00248 | 0.464 | +0.62 | −0.80 | **FAIL** |
| ln_ee | +0.00005 | 0.990 | +0.11 | +0.16 | **FAIL** |
| renew_share | +0.00128 | 0.778 | −0.69 | −0.17 | **FAIL** |
| solar_share | −0.00073 | 0.584 | +0.61 | −0.43 | **FAIL** |
| wind_share | −0.00068 | 0.869 | −0.19 | −0.54 | **FAIL** |
| new_renew_mw | −12.34 | 0.484 | +1.34 | +0.16 | **FAIL** |
| fossil_to_renew | −0.299 | 0.325 | +1.49 | −0.32 | **FAIL** |
| coal_share | −0.00131 | 0.779 | +0.34 | −0.19 | **FAIL** |

**All 8 outcomes fail the proper Lind-Mehlum test.** No slope is significant at both ends at the 10% one-sided level. The ln_ec inverted-U reported earlier (sign-only check: YES) required t > 1.645 at min and t < −1.645 at max; the actual t-values are 0.62 and −0.80. **There is no statistically significant evidence of nonlinearity for any outcome.** Turning-point estimates should not be cited.

---
## 5. Heterogeneity (2015-baseline moderators)

**Equation:** $Y_{it} = a_1 \text{AI}_{i,t-1} + \gamma_1 (\text{AI}_{i,t-1} \times D_i) + a_2 D_i + \text{controls} + \mu_i + \lambda_t$  
Moderators defined from 2015 values (fixed, cannot respond to AI). AI and D mean-centered.

In [ ]:
# Show renew_share interactions only (most informative)
print(het[outcome == 'renew_share', .(coef, estimate, se, pval, N)])

**Reading:**
- **AI × net_exporter (renew_share):** γ₁ = −0.019, p = 0.094. Net exporters show a more negative AI–renewable association. Borderline.
- **AI × high_price (renew_share):** γ₁ = **+0.032, p = 0.004.** High-electricity-price states show a significantly more positive AI–renewable association. Economically interpretable: expensive electricity creates stronger PPA incentives for data-center operators to source renewables.
- **AI × South:** γ₁ = +0.0002, p = 0.984. Null.
- **Exploratory label:** 9 interaction tests were run in total (3 moderators × 3 outcomes). The AI×high_price result is 1 of 9; it survives Bonferroni correction (0.05/9 = 0.0056). This is exploratory — EXPERIMENTS.md §4 did not specify a predicted sign for any interaction before Phase 4b ran.
- Note: time-invariant moderators (net_exporter_2015, high_price_2015) are absorbed by state FE and dropped as collinear; the interaction term is identified from time-varying AI × frozen baseline dummy.

---
## 6. Dynamics (t-1 only + long difference)

**Note:** QCEW 2012–2013 absent; t-2 and t-3 distributed lags omitted.

In [ ]:
print(dyn[, .(tag = sub('.*_ai518210_.*?_', '', tag), outcome, estimate, se, pval, N)])

**Reading:** Long-difference results (N=46 states; 4 states dropped for suppressed 2025 AI) are all insignificant. The ΔAI (2014→2024) has no detectable association with ΔY (2015→2025) in the cross-section. This is consistent with small within-state variation in AI: SD of ai_dc_per1000 ≈ 0.57, so 10-year changes are also small.

---
## 7. Georgia descriptive

In [ ]:
ga <- fread('results/2026-11-02/h1/georgia_descriptive.csv')
print(ga[, .(year, ai_dc_per1000, us_med_ai, renew_share, us_med_renew, resid_renew)])

**Reading:** Georgia's AI concentration rose from 2.0 to 5.0/1000 workers (2015–2025), well above the U.S. median (1.9–2.6). Georgia's renewable share rose from ~12% to ~13%, tracking close to the U.S. median (~20–28%). Residuals from the renew_share TWFE model are small (|ε| ≤ 0.02), meaning Georgia does not systematically over- or under-perform the model's prediction.

**NAICS 518210 caveat:** Georgia's high AI rank likely reflects AWS, Microsoft Azure, and Google data centers that serve commercial cloud workloads, not primarily AI training. This means ai_dc_per1000 overstates AI-specific compute intensity for Georgia.

---
## 9. new_renew_mw — robustness checks

**Purpose:** Test whether +74.5 MW survives scaling and influence analysis.  
**Within-state SD of AI:** 0.57 (use for "1-SD move" interpretation; pooled SD is 1.22).  
**Driver:** `code/R/04b_step1_twfe_checks.R`; results appended to `h1_extras.csv`.

**Reading:**

**Within-state SD of lagged AI: 0.57** (pooled SD: 1.22). The 1-SD within-state move is associated with ~42.5 MW more annual new renewable capacity (0.57 × 74.5 MW).

| Check | a₁ | SE | p | N | |
|---|---|---|---|---|---|
| Baseline (main result) | +74.5 MW | 31.5 | 0.022 | 550 | Reference |
| Per 1,000 MW 2015 capacity | +1.14 MW / 1,000 MW | 1.93 | 0.558 | 550 | Not significant when scaled |
| Per million people (2015 pop) | +12.8 MW / mpop | 8.86 | 0.156 | 550 | Not significant when scaled |
| Drop TX (highest new_renew_mw state) | +72.0 MW | 31.0 | 0.024 | 539 | Significant; TX does not drive result |
| Leave-one-state-out | min +58.0 (drop NC) | — | — | 539 | Range [+58, +91], entirely positive |
| | max +90.8 (drop WA) | — | — | 539 | Mean 74.5, SD 5.3 — very stable |

**Reading:** The scaled versions are not significant — large states (TX, WA) contribute both more absolute capacity and more AI employment, so dividing by size removes the leverage. The drop-TX result (N=539, p=0.024) confirms TX is not driving the significance. The LOSO range [+58, +91] is entirely positive with very low variance (SD=5.3) — no single state is a high-leverage outlier. The result is stable.

---
## 10. Generation channels

**Equation:** $\ln Y_{it} = \alpha_1 \cdot \text{AI}_{i,t-1} + \beta X_{it} + \mu_i + \lambda_t$  where $Y \in \{\text{gen\_renew\_mwh}, \text{gen\_fossil\_mwh}, \text{gen\_total\_mwh}\}$  
**Test:** Does AI predict renewable generation in MWh (not just capacity)? Explains why renew_share may fall while new_renew_mw rises.  
**Driver:** `code/R/04b_step1_twfe_checks.R`; results appended to `h1_extras.csv`.

**Reading:**

All three generation variables are strictly positive (min gen_renew_mwh = 114,097 MWh; min gen_fossil_mwh = 2,880 MWh; no zeros or negatives). Safe to log.

| Outcome | a₁ | SE | p | Interpretation |
|---|---|---|---|---|
| ln_renew_mwh | **−0.069** | 0.034 | **0.048** | More AI → significantly **less** renewable generation (MWh) |
| ln_fossil_mwh | +0.008 | 0.021 | 0.716 | Null |
| ln_total_mwh | −0.002 | 0.019 | 0.919 | Null |

**Pattern:** AI data-center concentration is associated with significantly less renewable *generation* (MWh), even as it predicts more new *capacity* (new_renew_mw, p=0.022). Total generation and fossil generation are unchanged.

**Likely explanations:**
1. **Timing mismatch:** New capacity (new_renew_mw) is added late in year t and has not yet generated substantial MWh in year t. The lagged-AI coefficient on new_renew_mw reflects a forward-looking signal; ln_renew_mwh captures the current stock.
2. **Composition:** High-AI states (WA, GA, VA) have large nuclear/gas/hydro baselines. Even as they build new renewable capacity, their existing generation mix remains fossil-heavy.

**Implication:** new_renew_mw and ln_renew_mwh describe different quantities — capacity additions (flow) vs. total generation (stock). Peril in current generation; possible blessing in the future capacity pipeline.

---
## 8. Figures

![H1 coefficient plot](../results/2026-11-02/figures/h1_coef_plot.png)

![Georgia AI and renewable paths](../results/2026-11-02/figures/georgia_ai_renew_paths.png)

---
## Summary

| Outcome | H1 verdict | Note |
|---|---|---|
| ln_ec (consumption) | Does not support H1 (p=0.704, CL) | No evidence of nonlinearity (all LM tests FAIL) |
| ln_ee (efficiency) | Does not support H1 (p=0.567, CL) | DK p=0.028 — sensitivity check only (T=11) |
| renew_share | Does not support H1 (p=0.103, CL) | Negative (peril direction); DK p=0.001 — sensitivity check only |
| new_renew_mw | **Supports H1 (p=0.022, CL)** | +74.5 MW/unit AI; LOSO range [+58, +91]; drop-TX p=0.024 |
| ln_renew_mwh | Does not support H1 (p=0.048, CL) | **Negative** — more AI → less renewable generation (MWh) |
| solar/wind share | Null | — |
| fossil_to_renew | Wrong sign (p=0.110) | — |
| coal_share | Null (falsification passes) | — |
| CO2/MWh | Null | — |

**Generation channels:** ln_renew_mwh is significantly negative (p=0.048); ln_total_mwh and ln_fossil_mwh are null. States with more AI data centers add renewable capacity but currently generate less renewable MWh — consistent with a timing mismatch (new capacity not yet generating) or composition effect (high-AI states have large fossil/nuclear baselines).

**Heterogeneity:** AI×high_price on renew_share γ₁=+0.032, p=0.004. Exploratory; 1 of 9 interaction tests; survives Bonferroni (0.05/9=0.0056).

**DK caveat:** With T=11, all DK results are sensitivity checks only. Baseline for every conclusion: clustered by state (CRV1).

**Nonlinearity:** All 8 proper Lind-Mehlum tests FAIL. No statistically significant evidence of nonlinearity for any outcome.

**Next:** `prompts/04c_step2_mediation.md`